<a href="https://colab.research.google.com/github/TonyQ2k3/pytorch-training/blob/main/notebooks/pytorch_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch Day 2
---

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
print(torch.__version__)

## Pytorch workflow
1. Get data ready and transform into tensors
2. Build / choose a model to train, pick a loss function, optimizer and training loop (epoch)
3. Fit the model to make predictions
4. Evaluate model
5. Improve the model if needed
6. Save it for later

### Data loading and preparing
Data can comes in different forms: text, images, videos, spreadsheets, audio, etc.

ML is all about transforming data into numerical representation, then have a model discover patterns in those representations.

In [ ]:
weight = 0.7
bias = 0.3

start = 0
end = 1
step = 0.02
x = torch.arange(start, end, step)
y = x * weight + bias

print(f"X set: {x}")
print(f"Y set: {y}")

### Making training and testing set

In [ ]:
training_index = int(0.8 * len(x))
x_train, y_train = x[:training_index], y[:training_index]
x_test, y_test = x[training_index:], y[training_index:]

In [ ]:
def plot_prediction(train_datas=x_train,
                    train_labels=y_train,
                    test_datas=x_test,
                    test_labels=y_test,
                    prediction=None):
  plt.figure(figsize=(10, 7))
  # Create scatter plot of training data
  plt.scatter(train_datas, train_labels, c="b", s=4, label="Training data")
  # Create scatter plot of testing data
  plt.scatter(test_datas, test_labels, c="g", s=4, label="Testing data")

  if prediction is not None:
    plt.scatter(test_datas, prediction, c="r", s=4, label="Predictions")

  plt.legend(prop={"size": 14})

In [ ]:
plot_prediction()

## Building a model
All neural network models inherit from a class called `nn.Module`.

+ `nn.Parameter` is a Tensor subclass, when used inside a `Module` it will be considered an attribute andd can be accessed in the `.parameters()` iterator.
+ `torch.randn()` returns a tensor filled with random numbers generated from a normal distribution.

The model will start out with 2 parameters: `weight` and `bias` as random numbers. Throughout the training with given data, the model will gradually update these parameters in order to return predictions closer to the labels.

What algorithms are used to update the parameters?
1. Gradient descent
2. Backpropagation


In [ ]:
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float))
    self.bias = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float))

  # forward() defines how input data will be transformed through the layers of the model to produce an output
  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.weight * x + self.bias

## Making predictions with `torch.inference_mode()`
`inference_mode()` is a context manager to be used when not training model. It disables gradient tracking to allow better performance.

In [ ]:
# Checking test dataset
print(x_test)
print(y_test)

In [ ]:
# Creating a model instance

torch.manual_seed(42)
my_model = LinearRegressionModel()

# Checking model's params
print(my_model.state_dict())

In [ ]:
with torch.inference_mode():
  y_preds = my_model(x_test)

print(y_preds)
# Very poor representation of data, aka bad prediction
plot_prediction(prediction=y_preds)

## Training a model

### Loss function
* Loss function is a function that quantifies the difference between predictions and the true values. Loss function takes in the prediction value of the model, and the target value, then return a value that represents the difference between them.

* The goal of training a ML model is to minimize the loss function, via the optimization function.

### Optimization algorithm
* Optimization algorithm is an algo used to optimize the parameters inside the model, in order to produce the best prediction possible.

### Training loop
`model.train()` puts a model into training mode.

When in training mode, certain functionalities get enabled:
+ Start tracking the gradients of model parameters for backpropagation
+ Dropout and Batch normalization layers get activated

**The training loop goes as follows**:
1. Create prediction (forward pass)
2. Use loss function to calculate the errors
3. Clear gradients from previous epoch (accumulated by default)
4. Compute the gradient of the loss
5. Update model parameters

In [ ]:
# Pick a loss function
loss_fn = nn.L1Loss()

# Pick an optimizer
optimizer = torch.optim.SGD(params=my_model.parameters(), lr=0.01)

# Setting the epoch
epochs = 100

# Put model into training mode
my_model.train()

# Start training loop
for epoch in range(epochs):

  # Create prediction
  y_pred = my_model(x_train)

  # Calculate loss
  loss = loss_fn(y_pred, y_train)

  # Optimizer zero grad
  optimizer.zero_grad()

  # Loss backward
  loss.backward()

  # Step the optimizer (Perform gradient descent to update params)
  optimizer.step()

  # Print out what's happening every 10 epochs
  if epoch % 10 == 0:
    print(f"Epoch: {epoch} | Loss: {loss}")


In [ ]:
# Checking model params
print(my_model.state_dict())

# Put model into evaluation mode
my_model.eval()

# Make predictions with the model
with torch.inference_mode():
  y_preds = my_model(x_test)

# The predictions (y_preds) should get closer to testing labels (y_test)
plot_prediction(prediction=y_preds)

## Wrap Up

### Creating, training and evaluating a model
1. Create datasets: training, testing (and validating) set
2. Define a function used for visualizing relationship between data & labels, as well as compare predictions and testing labels.
3. Create a model inheriting from `nn.Module` and define its parameters using `nn.Parameter()`. Remember to implement its `forward()` method.
4. Choose a loss function, an optimizer and an epoch.
5. Put model into training mode and start training:
  + Make predictions
  + Calculate `loss` between predictions and labels from training set
  + Clear optimizer gradient from previous epoch: `optimizer.zero_grad()`
  + Calculate loss gradient: `loss.backward()`
  + Update model parameters: `optimizer.step()`
6. Evaluate the model